In [3]:
import numpy as np

class DynamicPortfolio:
    def __init__(self, total_account, initial_funding_fraction=0.5):
        """
        total_account: total available capital (e.g., 1,000,000)
        initial_funding_fraction: fraction of total capital initially available for buys.
        """
        self.total_account = total_account
        # Funding cash is used exclusively to fund buy orders.
        self.funding_cash = total_account * initial_funding_fraction  
        # Liquidation cash comes from sell orders; not used to fund buys.
        self.liquidation_cash = total_account - self.funding_cash  
        self.positions = {}  # Tracks dollar positions in each asset.

    def update_portfolio(self, signals, allocation_factor=0.1, threshold=0.6):
        """
        Update portfolio positions based on signals.
        
        signals: dict mapping asset symbol to (buy_prob, sell_prob)
        allocation_factor: scales the dollar amount per unit probability (as a fraction of total_account)
        threshold: minimum probability required to trigger a buy or sell
        """
        for asset, (buy_prob, sell_prob) in signals.items():
            current_position = self.positions.get(asset, 0)
            
            # Process buy signals:
            if buy_prob > threshold:
                # Compute desired increase: higher buy probability implies a larger allocation.
                desired_increase = buy_prob * allocation_factor * self.total_account
                # Use only funding_cash to fund buys.
                if self.funding_cash >= desired_increase:
                    self.positions[asset] = current_position + desired_increase
                    self.funding_cash -= desired_increase
                    print(f"Bought ${desired_increase:,.2f} of asset {asset}.")
                else:
                    # If not enough funding cash, invest whatever is available.
                    self.positions[asset] = current_position + self.funding_cash
                    print(f"Bought ${self.funding_cash:,.2f} of asset {asset} (all available funding cash).")
                    self.funding_cash = 0

            # Process sell signals:
            elif sell_prob > threshold and current_position > 0:
                # Compute desired reduction.
                desired_reduction = sell_prob * allocation_factor * self.total_account
                reduction = min(desired_reduction, current_position)
                self.positions[asset] = current_position - reduction
                # Proceeds from sales go to the liquidation cash pool.
                self.liquidation_cash += reduction
                print(f"Sold ${reduction:,.2f} of asset {asset}.")
            else:
                print(f"Held asset {asset}.")
    
    def portfolio_status(self):
        """Return a summary of positions and cash balances."""
        total_invested = sum(self.positions.values())
        total_value = total_invested + self.funding_cash + self.liquidation_cash
        return {
            "positions": self.positions,
            "funding_cash": self.funding_cash,
            "liquidation_cash": self.liquidation_cash,
            "total_invested": total_invested,
            "total_account_value": total_value
        }

In [8]:
# Simulate 10 time steps with predefined probability signals.
if __name__ == '__main__':
    assets = ['A', 'B', 'C', 'D', 'E']
    portfolio = DynamicPortfolio(total_account=1_000_000, initial_funding_fraction=0.05)
    
    # Define signals for 10 time steps.
    # Each dictionary contains signals for each asset in the form (buy_prob, sell_prob)
    time_steps_signals = [
        {'A': (0.7, 0.1), 'B': (0.3, 0.6), 'C': (0.5, 0.2), 'D': (0.8, 0.3), 'E': (0.2, 0.4)},
        {'A': (0.4, 0.2), 'B': (0.7, 0.2), 'C': (0.6, 0.1), 'D': (0.3, 0.8), 'E': (0.5, 0.2)},
        {'A': (0.8, 0.1), 'B': (0.2, 0.7), 'C': (0.4, 0.2), 'D': (0.9, 0.1), 'E': (0.3, 0.5)},
        {'A': (0.2, 0.8), 'B': (0.6, 0.3), 'C': (0.7, 0.2), 'D': (0.4, 0.6), 'E': (0.5, 0.1)},
        {'A': (0.9, 0.1), 'B': (0.3, 0.7), 'C': (0.2, 0.8), 'D': (0.8, 0.2), 'E': (0.4, 0.6)},
        {'A': (0.3, 0.7), 'B': (0.8, 0.1), 'C': (0.6, 0.2), 'D': (0.2, 0.9), 'E': (0.7, 0.1)},
        {'A': (0.5, 0.2), 'B': (0.4, 0.3), 'C': (0.8, 0.1), 'D': (0.6, 0.2), 'E': (0.3, 0.7)},
        {'A': (0.2, 0.8), 'B': (0.9, 0.1), 'C': (0.4, 0.6), 'D': (0.3, 0.7), 'E': (0.6, 0.2)},
        {'A': (0.7, 0.1), 'B': (0.3, 0.8), 'C': (0.5, 0.2), 'D': (0.4, 0.6), 'E': (0.8, 0.1)},
        {'A': (0.3, 0.7), 'B': (0.5, 0.2), 'C': (0.6, 0.2), 'D': (0.7, 0.1), 'E': (0.4, 0.6)},
    ]
    
    # Loop over each time step, update the portfolio, and display the status.
    for t, signals in enumerate(time_steps_signals, start=1):
        print(f"\nTime Step {t}:")
        portfolio.update_portfolio(signals)
        status = portfolio.portfolio_status()
        print("Portfolio status after this time step:")
        for key, value in status.items():
            print(f"  {key}: {value}")



Time Step 1:
Bought $50,000.00 of asset A (all available funding cash).
Held asset B.
Held asset C.
Bought $0.00 of asset D (all available funding cash).
Held asset E.
Portfolio status after this time step:
  positions: {'A': 50000.0, 'D': 0}
  funding_cash: 0
  liquidation_cash: 950000.0
  total_invested: 50000.0
  total_account_value: 1000000.0

Time Step 2:
Held asset A.
Bought $0.00 of asset B (all available funding cash).
Held asset C.
Held asset D.
Held asset E.
Portfolio status after this time step:
  positions: {'A': 50000.0, 'D': 0, 'B': 0}
  funding_cash: 0
  liquidation_cash: 950000.0
  total_invested: 50000.0
  total_account_value: 1000000.0

Time Step 3:
Bought $0.00 of asset A (all available funding cash).
Held asset B.
Held asset C.
Bought $0.00 of asset D (all available funding cash).
Held asset E.
Portfolio status after this time step:
  positions: {'A': 50000.0, 'D': 0, 'B': 0}
  funding_cash: 0
  liquidation_cash: 950000.0
  total_invested: 50000.0
  total_account_v